# Data Cleaning con Pandas

> Pandas es una librería de Python escrita como extensión de NumPy para manipulación y análisis de datos.

Sitio oficial: [pandas.python.org](https://pandas.pydata.org/)
Documentación Oficial: [pandas.pydata.org/pandas-docs/stable/](https://pandas.pydata.org/pandas-docs/stable/)

In [1]:
import pandas as pd

## Indexando DataFrames

Tanto las Series como los DataFrames pueden tener índices.

Un índice es esencialmente una etiqueta a nivel de fila, y en pandas, las filas corresponden al eje cero. Los índices pueden autogenerarse, por ejemplo, al crear una nueva Serie sin índice, en cuyo caso obtenemos valores numéricos, o pueden establecerse explícitamente, como al usar un diccionario para crear la Serie o al cargar datos desde un archivo CSV y establecer los parámetros adecuados. Otra opción para establecer un índice es usar la función `set_index()`. Esta función toma una lista de columnas y las convierte en índices.

> La función `set_index()` es un proceso destructivo y no conserva el índice actual.
>
> Si desea conservar el índice actual, debe crear manualmente una nueva columna y copiar en ella los valores del atributo de índice.

En este notebook, exploraremos con más detalle cómo funcionan los índices en pandas.

### set_index()

In [2]:
df = pd.read_csv("../data/admission-predict.csv", index_col=0)

df.head()

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
Serial No.,,,,,,,,
1,337,118,4,4.5,4.5,9.65,1,0.92
2,324,107,4,4.0,4.5,8.87,1,0.76
3,316,104,3,3.0,3.5,8.00,1,0.72
4,322,110,3,3.5,2.5,8.67,1,0.80
5,314,103,2,2.0,3.0,8.21,0,0.65


Supongamos que no queremos indexar el DataFrame por números de serie, sino por probabilidad de admisión. Sin embargo, asumamos que queremos conservar el número de serie para usarlo posteriormente. Para ello, guardemos el número de serie en una nueva columna. Podemos hacerlo utilizando el operador de indexación en la cadena que contiene la etiqueta de la columna. Luego, podemos usar `set_index` para asignar el índice de la columna a la probabilidad de admisión.

De esta forma, copiamos los datos indexados a su propia columna.

In [3]:
df['Serial Number'] = df.index

# Configuramos como índice otra columna
df = df.set_index('Chance of Admit ')

df.head()

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Serial Number
Chance of Admit,,,,,,,,
0.92,337,118,4,4.5,4.5,9.65,1,1
0.76,324,107,4,4.0,4.5,8.87,1,2
0.72,316,104,3,3.0,3.5,8.00,1,3
0.80,322,110,3,3.5,2.5,8.67,1,4
0.65,314,103,2,2.0,3.0,8.21,0,5


Verás que al crear un nuevo índice a partir de una columna existente, el índice tiene un nombre, que es el nombre original de la columna.

Podemos eliminar el índice por completo llamando a la función `reset_index()`. Esto convierte el índice en una columna y crea un índice numerado por defecto.

In [4]:
df = df.reset_index()
df.head()

,Chance of Admit,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Serial Number
0,0.92,337,118,4,4.5,4.5,9.65,1,1
1,0.76,324,107,4,4.0,4.5,8.87,1,2
2,0.72,316,104,3,3.0,3.5,8.00,1,3
3,0.80,322,110,3,3.5,2.5,8.67,1,4
4,0.65,314,103,2,2.0,3.0,8.21,0,5


### Indexación multinivel

Una característica interesante de Pandas es la indexación multinivel. Es similar a las claves compuestas en los sistemas de bases de datos relacionales. Para crear un índice multinivel, simplemente llamamos a `set index` y le proporcionamos una lista de columnas que queremos indexar.

Pandas buscará en estas columnas en orden, encontrando los datos distintos y formando índices compuestos. Un buen ejemplo de esto se encuentra a menudo al trabajar con datos geográficos ordenados por regiones o datos demográficos.

Cambiemos de conjunto de datos y veamos algunos datos censales para un mejor ejemplo. Estos datos se almacenan en el archivo `census.csv` y provienen de la Oficina del Censo de los Estados Unidos. En particular, se trata de un desglose de los datos de población a nivel de condado en EE. UU. Es un excelente ejemplo de cómo se pueden formatear diferentes tipos de conjuntos de datos al limpiarlos.

Importemos los datos y veamos cómo se ven.

In [5]:
df = pd.read_csv('../data/census.csv')
df.head()

,SUMLEV,REGION,DIVISION,STATE,COUNTY,STNAME,CTYNAME,CENSUS2010POP,ESTIMATESBASE2010,POPESTIMATE2010,...,RDOMESTICMIG2011,RDOMESTICMIG2012,RDOMESTICMIG2013,RDOMESTICMIG2014,RDOMESTICMIG2015,RNETMIG2011,RNETMIG2012,RNETMIG2013,RNETMIG2014,RNETMIG2015
0,40,3,6,1,0,Alabama,Alabama,4779736,4780127,4785161,...,0.002295,-0.193196,0.381066,0.582002,-0.467369,1.030015,0.826644,1.383282,1.724718,0.712594
1,50,3,6,1,1,Alabama,Autauga County,54571,54571,54660,...,7.242091,-2.915927,-3.012349,2.265971,-2.530799,7.606016,-2.626146,-2.722002,2.592270,-2.187333
2,50,3,6,1,3,Alabama,Baldwin County,182265,182265,183193,...,14.832960,17.647293,21.845705,19.243287,17.197872,15.844176,18.559627,22.727626,20.317142,18.293499
3,50,3,6,1,5,Alabama,Barbour County,27457,27457,27341,...,-4.728132,-2.500690,-7.056824,-3.904217,-10.543299,-4.874741,-2.758113,-7.167664,-3.978583,-10.543299
4,50,3,6,1,7,Alabama,Bibb County,22915,22919,22861,...,-5.527043,-5.068871,-6.201001,-0.177537,0.177258,-5.088389,-4.363636,-5.403729,0.754533,1.107861


En este conjunto de datos hay dos niveles resumidos: uno que contiene datos resumidos para todo el país y otro que contiene datos resumidos para cada estado.

Quiero ver una lista de todos los valores únicos en una columna determinada. En este DataFrame, vemos que los valores posibles para el nivel de suma se obtienen usando la función `unique`. Esto es similar al operador `distinct` de SQL.

Aquí podemos aplicar `unique` al nivel de suma de nuestro DataFrame actual.

In [6]:
df['SUMLEV'].unique()

array([40, 50])

> Vemos que solo hay dos valores diferentes: 40 y 50.

Excluyamos todas las filas que sean resúmenes (SUMLEV = 50, summarize level) a nivel estatal y conservemos solo los datos del condado.

In [7]:
df=df[df['SUMLEV'] == 50]
df.head()

,SUMLEV,REGION,DIVISION,STATE,COUNTY,STNAME,CTYNAME,CENSUS2010POP,ESTIMATESBASE2010,POPESTIMATE2010,...,RDOMESTICMIG2011,RDOMESTICMIG2012,RDOMESTICMIG2013,RDOMESTICMIG2014,RDOMESTICMIG2015,RNETMIG2011,RNETMIG2012,RNETMIG2013,RNETMIG2014,RNETMIG2015
1,50,3,6,1,1,Alabama,Autauga County,54571,54571,54660,...,7.242091,-2.915927,-3.012349,2.265971,-2.530799,7.606016,-2.626146,-2.722002,2.592270,-2.187333
2,50,3,6,1,3,Alabama,Baldwin County,182265,182265,183193,...,14.832960,17.647293,21.845705,19.243287,17.197872,15.844176,18.559627,22.727626,20.317142,18.293499
3,50,3,6,1,5,Alabama,Barbour County,27457,27457,27341,...,-4.728132,-2.500690,-7.056824,-3.904217,-10.543299,-4.874741,-2.758113,-7.167664,-3.978583,-10.543299
4,50,3,6,1,7,Alabama,Bibb County,22915,22919,22861,...,-5.527043,-5.068871,-6.201001,-0.177537,0.177258,-5.088389,-4.363636,-5.403729,0.754533,1.107861
5,50,3,6,1,9,Alabama,Blount County,57322,57322,57373,...,1.807375,-1.177622,-1.748766,-2.062535,-1.369970,1.859511,-0.848580,-1.402476,-1.577232,-0.884411


Si bien este conjunto de datos es interesante por varias razones, vamos a reducir el análisis a las estimaciones de población total y el número total de nacimientos.

Para ello, crearemos una lista con los nombres de las columnas que queremos conservar, proyectaremos esos valores y asignaremos el DataFrame resultante a nuestra variable `df`.

In [8]:
columns_to_keep = ['STNAME','CTYNAME','BIRTHS2010','BIRTHS2011','BIRTHS2012','BIRTHS2013',
                   'BIRTHS2014','BIRTHS2015','POPESTIMATE2010','POPESTIMATE2011',
                   'POPESTIMATE2012','POPESTIMATE2013','POPESTIMATE2014','POPESTIMATE2015']
df = df[columns_to_keep]
df.head()

,STNAME,CTYNAME,BIRTHS2010,BIRTHS2011,BIRTHS2012,BIRTHS2013,BIRTHS2014,BIRTHS2015,POPESTIMATE2010,POPESTIMATE2011,POPESTIMATE2012,POPESTIMATE2013,POPESTIMATE2014,POPESTIMATE2015
1,Alabama,Autauga County,151,636,615,574,623,600,54660,55253,55175,55038,55290,55347
2,Alabama,Baldwin County,517,2187,2092,2160,2186,2240,183193,186659,190396,195126,199713,203709
3,Alabama,Barbour County,70,335,300,283,260,269,27341,27226,27159,26973,26815,26489
4,Alabama,Bibb County,44,266,245,259,247,253,22861,22733,22642,22512,22549,22583
5,Alabama,Blount County,183,744,710,646,618,603,57373,57711,57776,57734,57658,57673


Los datos del Censo de EE. UU. desglosan las estimaciones de población por estado y condado. Podemos cargar los datos y establecer el índice combinando los valores de estado y condado para ver cómo pandas los maneja en un DataFrame.

Para ello, creamos una lista con los identificadores de columna que queremos indexar. Luego, llamamos a `set_index()` con esta lista y asignamos la salida según corresponda. Aquí vemos que tenemos un índice doble: primero, el nombre del estado y, segundo, el nombre del condado.

In [9]:
df = df.set_index(['STNAME', 'CTYNAME'])
df.head()

BIRTHS2010  BIRTHS2011  BIRTHS2012  BIRTHS2013  \
STNAME  CTYNAME                                                          
Alabama Autauga County         151         636         615         574   
        Baldwin County         517        2187        2092        2160   
        Barbour County          70         335         300         283   
        Bibb County             44         266         245         259   
        Blount County          183         744         710         646   

                        BIRTHS2014  BIRTHS2015  POPESTIMATE2010  \
STNAME  CTYNAME                                                   
Alabama Autauga County         623         600            54660   
        Baldwin County        2186        2240           183193   
        Barbour County         260         269            27341   
        Bibb County            247         253            22861   
        Blount County          618         603            57373   

                        POPESTIMATE2011  POPESTIMATE2012  POPESTIMATE2013  \
STNAME  CTYNAME                                                             
Alabama Autauga County            55253            55175            55038   
        Baldwin County           186659           190396           195126   
        Barbour County            27226            27159            26973   
        Bibb County               22733            22642            22512   
        Blount County             57711            57776            57734   

                        POPESTIMATE2014  POPESTIMATE2015  
STNAME  CTYNAME                                           
Alabama Autauga County            55290            55347  
        Baldwin County           199713           203709  
        Barbour County            26815            26489  
        Bibb County               22549            22583  
        Blount County             57658            57673

Una pregunta inmediata que surge es cómo podemos consultar este DataFrame.

Ya vimos que el atributo `loc` del DataFrame puede recibir múltiples argumentos y que permite consultar tanto las filas como las columnas. Al usar un MultiIndex, es necesario proporcionar los argumentos en orden según el nivel de consulta deseado. Dentro del índice, cada columna se denomina nivel, y la columna más externa es el nivel cero.

Si queremos ver los resultados de población del condado de 'Washtenaw County' en 'Michigan' el primer argumento sería "Michigan" y el segundo, 'Washtenaw County'.

In [10]:
df.loc['Michigan', 'Washtenaw County']

BIRTHS2010            977
BIRTHS2011           3826
BIRTHS2012           3780
BIRTHS2013           3662
BIRTHS2014           3683
BIRTHS2015           3709
POPESTIMATE2010    345563
POPESTIMATE2011    349048
POPESTIMATE2012    351213
POPESTIMATE2013    354289
POPESTIMATE2014    357029
POPESTIMATE2015    358880
Name: (Michigan, Washtenaw County), dtype: int64

Si desea comparar dos condados, por ejemplo, 'Washtenaw County' y 'Wayne County', puede pasar a `loc` una lista de tuplas que describan los índices que desea consultar. Dado que tenemos un MultiIndex con dos valores (el estado y el condado), necesitamos proporcionar dos valores como cada elemento de nuestra lista de filtrado. Cada tupla debe tener dos elementos: el primero es el primer índice y el segundo, el segundo.

Por lo tanto, en este caso, tendremos una lista de dos tuplas. En cada tupla, el primer elemento es Michigan y el segundo es el condado de 'Washtenaw County' o 'Wayne County'.

In [11]:
df.loc[ [('Michigan', 'Washtenaw County'),
         ('Michigan', 'Wayne County')] ]

BIRTHS2010  BIRTHS2011  BIRTHS2012  BIRTHS2013  \
STNAME   CTYNAME                                                            
Michigan Washtenaw County         977        3826        3780        3662   
         Wayne County            5918       23819       23270       23377   

                           BIRTHS2014  BIRTHS2015  POPESTIMATE2010  \
STNAME   CTYNAME                                                     
Michigan Washtenaw County        3683        3709           345563   
         Wayne County           23607       23586          1815199   

                           POPESTIMATE2011  POPESTIMATE2012  POPESTIMATE2013  \
STNAME   CTYNAME                                                               
Michigan Washtenaw County           349048           351213           354289   
         Wayne County              1801273          1792514          1775713   

                           POPESTIMATE2014  POPESTIMATE2015  
STNAME   CTYNAME                                             
Michigan Washtenaw County           357029           358880  
         Wayne County              1766008          1759335

### Conclusiones:

Así es como funcionan los índices jerárquicos en pocas palabras. Son una parte especial de la librería pandas que facilita la gestión y el análisis de datos.

Por supuesto, el etiquetado jerárquico no se limita a las filas. Por ejemplo, puedes transponer esta matriz y obtener etiquetas jerárquicas en las columnas. Y proyectar una sola columna con estas etiquetas funciona tal como cabría esperar.